# 10. Evaluate the profile and compare classifiers

We will combine matched errors into source-level profiles, freeze the normal reference, compare methods on the same sources, and keep supervised condition recognition separate from anomaly detection.

## Run this notebook on its own

From `experiments/aaai`, run `uv sync`, then `uv run jupyter lab`. Select the Python environment created here. Python 3.12, PyTorch, pandas, scikit-learn, matplotlib, python-dotenv, the notebook libraries, and the local `ambient` package are declared in `pyproject.toml`. Each notebook imports and loads everything it needs.

All configuration and credentials remain in the repository root `.env`. Set `AAAI_DEMO=1` for artificial arrays used to check the software, or `AAAI_DEMO=0` for available real poses. Restart the kernel after changing settings. Artificial examples have no medical meaning. The setup uses aliases for python-dotenv's actual function names so that `load_env(find_env())` works.

The video below provides real walking context in either mode. It is separate from any artificial example. These research notebooks do not launch a large download or full experiment without an explicit command.

## Historical tutorial and current study

This notebook preserves the earlier tutorial's code, observations and experimental scope. Its original result identities remain historical; they are not measurements of the new retained-predictor or U/J/A study. Start the current independent tutorial sequence at [Notebook 12](12_observation_population_and_video_context.ipynb), then continue through [Notebook 17](17_failure_analysis_and_manuscript_decision.ipynb). The [current protocol](docs/iteration02/protocol.md) and [current results](docs/iteration02/results.md) identify the verified new run. The [draft01 manuscript](docs/draft01/manuscript.md) remains the record of the completed earlier study.

The current primary question is normal versus the four requested condition families, with all other folders retained in a mandatory broad secondary analysis. It tests opposite-leg information beyond own-leg geometry and extra capacity. Uniform masking remains the reference, while J and A are declared motion-independent alternatives with matched budgets. Read the newer notebooks for their support rules and fixed evaluation banks. Do not combine old and new scores or denominators.

In [ ]:
from pathlib import Path
import os, sys, json
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'ambient').is_dir() and (p/'data/gavd').is_dir())
BASE=ROOT/'experiments/aaai'
os.chdir(ROOT)
from dotenv import load_dotenv as load_env, find_dotenv as find_env
load_env(find_env())
os.chdir(BASE)
if str(BASE) not in sys.path:sys.path.insert(0,str(BASE))
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display,Markdown
from aaai_gait.settings import Settings
from aaai_gait.study_data import build_study_manifest,make_outer_folds,fold_metadata
from aaai_gait.study_runner import prepare_study
from aaai_gait.media import show_video_context,plot_motion
settings=Settings();torch.set_num_threads(2)
fresh_manifest=build_study_manifest(ROOT,settings.pose_dir)
print('Mode:', 'SYNTHETIC SOFTWARE CHECK' if settings.demo else 'REAL GAVD, subject to quality exclusions')
print('Fresh annotations:',len(fresh_manifest),'sequences;',fresh_manifest.source_id.nunique(),'source videos')
video_row=show_video_context(fresh_manifest,frame_number_base=settings.frame_number_base)

## 1. Build a profile that a reader can explain

For each joint, subtract full-context error from own-joint error. Average common targets within a window, then windows within a sequence, then sequences within a source video. Give each of the four joints in a body region equal weight. The result has three numbers: left-leg help, right-leg help, and arm help.

This hierarchy prevents a long sequence from dominating a source. It also prevents a highly visible joint from dominating its region. At least four distinct physical patches must support every readout joint. Missing regions make a source unscorable; they never become zero-valued evidence of normality.

![Three regions and a normal reference](docs/draft00/images/support_profile.svg)

In [ ]:
# This matrix is invented solely to explain the reference calculation.
normal_profiles=np.array([[.10,.11,.06],[.12,.10,.05],[.09,.12,.07],[.11,.09,.06],[.10,.10,.05]])
example_profile=np.array([.03,.16,.06])
mean=normal_profiles.mean(0);variance=normal_profiles.var(0,ddof=1)
reference_variance=np.maximum(.5*variance+.5*variance.mean(),1e-8)
contribution=(example_profile-mean)**2/reference_variance
print('Illustrative normal-reference distance:',contribution.mean())
display(pd.DataFrame({'region':['left leg','right leg','arms'],'normal_mean':mean,
                      'example_help':example_profile,'squared_standardized_deviation':contribution}))

## 2. Estimate normal variation without touching the test sources

The five inner normal sources estimate the mean and variation of each region. Half of each region's sample variance is combined with half of the mean variance across regions. This shrinkage limits unstable estimates from a small reference group, but it does not solve the small-sample problem. A floor prevents division by zero. The anomaly score is the mean squared standardized distance across the three regions.

Every fold has its own fitted models and reference. Raw distances from different folds need not have the same scale. We calculate AUROC within each fold and average those values using the number of normal test sources as weights. We also report average precision with each fold's abnormal prevalence. A high average precision can be less informative when most videos are labeled abnormal.

![Separate the normal reference and test](docs/draft00/images/neuromechanics_experiment.svg)

In [ ]:
kind='synthetic' if settings.demo else 'gavd'
latest=BASE/'artifacts/context_study'/kind/'latest.json'
run_dir=None
if latest.exists():
    pointer=json.loads(latest.read_text());run_dir=Path(pointer['run_dir'])
    if not run_dir.is_absolute():run_dir=ROOT/run_dir
    summary=json.loads((run_dir/'summary.json').read_text())
    assert bool(summary['synthetic'])==settings.demo
    print('Recorded artifact:',run_dir,'status:',summary['status'])
    display(summary)
else:
    print('No completed artifact in this mode. Run scripts/run_context_study.py first.')
    print('The calculations below teach the protocol without inventing results.')

In [ ]:
if run_dir is not None:
    metrics=pd.read_csv(run_dir/'ranking_metrics.csv')
    key_methods=['trained_latent_all_profile_score','trained_latent_all_raw_full',
                 'trained_coordinate_all_profile_score','ridge_coordinate_all_profile_score',
                 'untrained_latent_all_profile_score']
    display(metrics[metrics.method.isin(key_methods)])
    display(json.loads((run_dir/'primary_paired_bootstrap.json').read_text()))
    print('Synthetic checks have no scientific interpretation.' if settings.demo else 'These are recorded real-run estimates, not a guarantee of generalization.')

## 3. Compare the proposed contribution with simpler explanations

The primary comparison is the trained latent profile against full-context latent error on the same twelve joints, masks, targets, and sources. The coordinate profile checks whether the effect remains with a local measurement target. Ridge regression checks whether a linear completion model already captures the effect. An untrained encoder checks whether representation learning is necessary.

The patch-context head asks whether simultaneous geometry provides most of the help. Separate normal-reference distances for own-joint and full-context errors test whether subtraction adds value. The fixed-budget sensitivity retains four physical patches per joint and source. Reference sensitivity removes one inner normal source at a time while keeping predictor weights fixed. These comparisons must not be selected because they produce the largest test score.

In [ ]:
if run_dir is not None:
    paired=pd.read_csv(run_dir/'paired_source_scores.csv')
    print('Unique outer-test sources:',paired.source_id.nunique())
    display(paired.groupby('outer_fold').binary_label.agg(['size','sum']))
    assert not paired.source_id.duplicated().any()
    files=sorted(run_dir.glob('fold_*/trained_reference_sensitivity.csv'))
    if files:
        sensitivity=pd.concat([pd.read_csv(p) for p in files],ignore_index=True)
        ranges=sensitivity.groupby(['outer_fold','target_type','source_id']).score.agg(lambda values: values.max()-values.min()).rename('reference_score_range')
        display(ranges.reset_index().groupby(['outer_fold','target_type']).reference_score_range.agg(['min','median','max']))

## 4. Keep uncertainty at the source level

The paired bootstrap resamples source IDs within each outer fold and binary label. Each sampled source carries both methods' scores. Windows do not become independent observations. The interval conditions on the fitted models, reference samples, and fold assignments. It omits training uncertainty, reference-selection uncertainty, and repeated-person dependence across source IDs.

A positive estimate with an interval spanning zero does not establish improvement. A negative result also matters: it would show that this context comparison did not add value under this protocol. Do not reverse the score or search for a favorable split after reading it.

## 5. Compare the five-condition classifiers fairly

The Random Forest uses exp5-style angle and motion summaries from the same normalized pose windows. The JEPA classifier uses the frozen normal-trained encoder followed by a linear classifier. Both receive normal and abnormal labels from non-test fitting sources, excluding the five normal reference sources. The encoder itself remains normal-trained.

These classifiers answer a supervised five-label question. They receive more label information than the normal-only anomaly detectors, so their macro-F1 scores belong in a separate table. Hyperparameters are fixed. Probabilities are averaged within sequences and then sources. A source with several focal labels needs explicit source-label handling and cannot be counted as independent people. Labels are dataset annotations, not verified diagnoses.

The full runner executes `condition_capstone` from `aaai_gait.study_baselines` in each fold. Notebook 06 retains the detailed earlier exp5 comparison and video-level prediction tutorial.

In [ ]:
if run_dir is not None:
    rows=[]
    for path in sorted(run_dir.glob('fold_*/condition_metrics.json')):
        data=json.loads(path.read_text())
        for name,value in data['metrics'].items():
            rows.append({'fold':path.parent.name,'method':name,'macro_f1':value['macro_f1'],
                         'balanced_accuracy':value['balanced_accuracy'],'test_sources':value['n_test_sources']})
    display(pd.DataFrame(rows))

## 6. Interpret condition-level results cautiously

Report each category's source count, supported folds, AUROC, average precision, and prevalence. Condition summaries reuse the same normal sources and are dependent. Categories with two or three sources are descriptive examples. Even a strong result could reflect age, camera, assistive devices, or uploader style. GAVD cannot establish performance on new clinical participants, sites, or disease stages without an independent cohort.

## Record what this experiment establishes

Keep the run protocol, source roles, exclusions, training curves, frozen reference parameters, and source scores together. A diagram explains an intended comparison. Only a completed real-data run supplies evidence for it. Read the [implemented protocol](docs/context_study_protocol.md), [current results](docs/context_study_results.md), and [review of the research direction](docs/draft00/README.md) before making a manuscript claim.